In [5]:
import dspy
from dspy.datasets import HotPotQA
from dspy.evaluate import Evaluate

In [17]:
lm = dspy.LM(
    "ollama_chat/llama3.2:latest",
    api_base="http://localhost:11434",
    api_key=""
)
colbertv2_wiki17_abstracts = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')
dspy.configure(lm=lm, rm=colbertv2_wiki17_abstracts)

In [18]:
lm("Say hello in one sentence.")

['Hello!']

In [2]:
dataset = HotPotQA(train_seed=1, train_size=3000, eval_seed=2023, dev_size=100, test_size=0)

README.md: 0.00B [00:00, ?B/s]

fullwiki/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/validation-00000-of-00001.parqu(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

fullwiki/test-00000-of-00001.parquet:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [3]:
trainset = [x.with_inputs('question') for x in dataset.train]
devset = [x.with_inputs('question') for x in dataset.dev]

len(trainset), len(devset)

dev_example = devset[1]
print(f"Question: {dev_example.question}")
print(f"Answer: {dev_example.answer}")
print(f"Relevant Wikipedia Titles: {dev_example.gold_titles}")

Question: Who conducts the draft in which Marc-Andre Fleury was drafted to the Vegas Golden Knights for the 2017-18 season?
Answer: National Hockey League
Relevant Wikipedia Titles: {'2017 NHL Expansion Draft', '2017–18 Pittsburgh Penguins season'}


In [6]:
metric_EM = dspy.evaluate.answer_exact_match
evaluate_hotpot = Evaluate(devset=devset, metric=metric_EM, num_threads=1, display_progress=True, display_table=0)

In [7]:
class GenerateAnswer(dspy.Signature):
    """Answer questions with short factoid answers."""

    context = dspy.InputField(desc="may contain relevant facts")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="often between 1 and 5 words")

In [9]:
class RAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()

        self.retrieve = dspy.Retrieve(k=num_passages)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)
    
    def forward(self, question):
        context = self.retrieve(question).passages
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

In [15]:
# Ask any question you like to this simple RAG program.
my_question = "What castle did David Gregory inherit?"

rag_mistral = RAG()
# Get the prediction. This contains `pred.context` and `pred.answer`.
pred = rag_mistral(my_question)

# Print the contexts and the answer.
print(f"Question: {my_question}")
print(f"Predicted Answer: {pred.answer}")
print(f"Retrieved Contexts (truncated): {[c[:200] + '...' for c in pred.context]}")

"""
Question: What castle did David Gregory inherit?
Predicted Answer: Kinnairdy Castle
Retrieved Contexts (truncated): ['David Gregory (physician) | David Gregory (20 December 1625 – 1720) was a Scottish physician and inventor. His surname is sometimes spelt as Gregorie, the original Scottish spelling. He inherited Kinn...', 'Gregory Tarchaneiotes | Gregory Tarchaneiotes (Greek: Γρηγόριος Ταρχανειώτης , Italian: "Gregorio Tracanioto" or "Tracamoto" ) was a "protospatharius" and the long-reigning catepan of Italy from 998 t...', 'David Gregory (mathematician) | David Gregory (originally spelt Gregorie) FRS (? 1659 – 10 October 1708) was a Scottish mathematician and astronomer. He was professor of mathematics at the University ...']
"""

ConnectTimeout: HTTPConnectionPool(host='20.102.90.50', port=2017): Max retries exceeded with url: /wiki17_abstracts?query=What+castle+did+David+Gregory+inherit%3F&k=3 (Caused by ConnectTimeoutError(<HTTPConnection(host='20.102.90.50', port=2017) at 0x16b8a9010>, 'Connection to 20.102.90.50 timed out. (connect timeout=10)'))

In [16]:
class GenerateAnswer(dspy.Signature):
    """Answer questions with short factoid answers."""

    question = dspy.InputField()
    answer = dspy.OutputField(desc="often between 1 and 5 words")


qa = dspy.Predict(GenerateAnswer)
pred = qa(question="What castle did David Gregory inherit?")
print(pred.answer)

BadRequestError: litellm.BadRequestError: GetLLMProvider Exception - list index out of range

original model: mistral